# turkish_distill: TR-MTEB evaluation

Evaluates any embedding model (teachers, baselines, our trained students) on TR-MTEB (`MTEB(Turkish)`, 26 tasks) and compares the finished runs. This notebook is only for evaluation; training lives in `colab_pipeline.ipynb`.

- Results go to Drive: `bitirme/eval/<label>/` (one folder per model, one JSON per task). An interrupted run resumes and skips finished tasks.
- Models are downloaded to the Colab local disk, never to Drive.
- Default protocol: plain encoding, **no instruction/prefix** (this is how the TR-MTEB authors evaluate). Prefixed variants are separate presets with their own labels.
- Use a **separate runtime** from the one doing teacher encoding: both need the GPU.

Before the first run: `HF_TOKEN` in Colab Secrets.

## 1. Setup

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'
EVAL_DIR = f'{DRIVE_ROOT}/eval'

os.makedirs(EVAL_DIR, exist_ok=True)
# HF cache on the LOCAL disk (Drive corrupts large model files).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
# The TR-MTEB authors' fork of MTEB. It pins datasets<3, so pip may downgrade `datasets` in this runtime.
!pip install --quiet "git+https://github.com/selmanbaysan/mteb_tr.git"
!git log -1 --oneline

Check that the Turkish benchmark is available (if this fails, restart the runtime once and re-run from the top):

In [ ]:
import mteb
print('mteb', getattr(mteb, '__version__', '?'))
bench = mteb.get_benchmark('MTEB(Turkish)')
print(len(list(bench.tasks)), 'tasks')

## 2. Model presets

Revisions are Hugging Face commit hashes fetched on 2026-10-11, so every run is reproducible. A preset is just the arguments of `run_evaluation`; add your own entries (e.g. a trained student: `model` = path of its `final/` folder on Drive).

Reference numbers for a sanity check (other papers/cards, their own protocols, so only roughly comparable): Qwen3-Embedding-8B 68.66, mE5-large-instruct 67.47, BGE-M3 67.36 (MoganBert paper, TR-MTEB mean over tasks); `modernbert-tr-embed` 68.14 and `turkish-e5-large` 67.17 (model card). A plain run that is far below these probably means a protocol difference (see the `*_instr` presets).

In [ ]:
INSTR = 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: '

PRESETS = {
    # --- teacher candidates ---
    'bge_m3': dict(model='BAAI/bge-m3', revision='5617a9f61b028005a4858fdac845db406aefb181', dtype='float16'),
    'me5_large_instruct': dict(model='intfloat/multilingual-e5-large-instruct', revision='274baa43b0e13e37fafa6428dbc7938e62e5c439', dtype='float16'),
    'me5_large_instruct_instr': dict(model='intfloat/multilingual-e5-large-instruct', revision='274baa43b0e13e37fafa6428dbc7938e62e5c439', dtype='float16', query_prompt=INSTR),
    'qwen3_8b': dict(model='Qwen/Qwen3-Embedding-8B', revision='1d8ad4ca9b3dd8059ad90a75d4983776a23d44af', dtype='bfloat16', batch_size=16),
    # Instruction format taken from memory of the Qwen3 model card: verify it before trusting this preset.
    'qwen3_8b_instr': dict(model='Qwen/Qwen3-Embedding-8B', revision='1d8ad4ca9b3dd8059ad90a75d4983776a23d44af', dtype='bfloat16', batch_size=16, query_prompt=INSTR),
    'turkish_e5_large': dict(model='ytu-ce-cosmos/turkish-e5-large', revision='02e2362d503bbdeafcb17143b2165c0743f9fdb1', dtype='float16'),
    # --- baselines from the same lab ---
    'modernbert_tr_embed': dict(model='ytu-ce-cosmos/modernbert-tr-embed', revision='855820e0079102805908d60d32cd9fd59c970b80'),
    'modernbert_tr_1k_embed': dict(model='ytu-ce-cosmos/modernbert-tr-1k-embed', revision='ba1e17bc7e95a257b4e614c300c4ca076711fce6'),
    # --- our student before any distillation (raw MLM encoder + mean pooling) ---
    'modernbert_tr_base_1k_raw': dict(model='ytu-ce-cosmos/modernbert-tr-base-1k', revision='c2e35f553101cce5b03fbb318ef8728473fd34dd'),
    # --- our trained students: point `model` at a run's final/ folder ---
    # 'my_student': dict(model=f'{DRIVE_ROOT}/runs/multi_a/<run_id>/final'),
}
print(list(PRESETS))

## 3. Run one evaluation

`TASKS = None` runs all 26 tasks. For a quick smoke test use a few small tasks; they are stored under a separate label (`<name>__subset`) so they never mix with full runs.

In [ ]:
CHOICE = 'bge_m3'
TASKS = None   # e.g. ['STSbTR', 'TurkishIronyClassification', 'TurkishMovieSentimentClassification']

from turkish_distill.eval.trmteb import run_evaluation, summarize_dir, format_table

label = CHOICE if TASKS is None else f'{CHOICE}__subset'
label_dir = run_evaluation(output_dir=EVAL_DIR, label=label, task_names=TASKS, **PRESETS[CHOICE])
print(format_table({label: summarize_dir(label_dir)}))

## 4. Compare all finished evaluations

In [ ]:
import glob
import pandas as pd

dirs = sorted(d for d in glob.glob(f'{EVAL_DIR}/*') if os.path.isdir(d) and not d.endswith('__subset'))
summaries = {os.path.basename(d): summarize_dir(d) for d in dirs}
print(format_table(summaries))

per_task = pd.DataFrame({k: v['per_task'] for k, v in summaries.items()}).round(2)
per_task